# NB_SLV_PN_PRODUCT_SALE

RECOMPUTE `lh_vv_silver.dbo.slv_pn_product_sale`. Một dòng mỗi `product_id`, kể cả product đã xoá. Giá NULL giữ NULL.


In [ ]:
p_exec_id = ""
p_run_id = ""
p_pl_name = ""
p_trg_fq = ""
p_pinned_json = ""
p_mode = "RUN"
p_force = "false"
p_history_limit = "1000"


In [ ]:
%run NB_LIB_TRANSFORM_SLV_GLD


In [ ]:
from pyspark import StorageLevel

FLAG = "_is_deleted"
OUT_COLS = (
    "product_id", "product_post_id", "original_price", "marketing_price", "markup_percent", "is_bookable",
)
PRODUCT_TYPES = {"id": "string", "metadata": "string", "deleted": "boolean"}
POST_TYPES = {
    "id": "string", "external_id": "string", "original_price": "string",
    "marketing_price": "string", "markup_percent": "string", "deleted": "boolean",
}
VARIANT_TYPES = {"product_id": "string", "status": "string", "deleted": "boolean"}

# BEGIN_PRICE_PURE
import base64
import binascii
import re

PRICE_NUM_RE = re.compile(r"^[+-]?(\d+(\.\d*)?|\.\d+)([eE][+-]?\d+)?$")


def decode_price(value, scale):
    """mapping_slv_pn_product_l2.md §1.3: NULL/boolean -> None; số giữ; chuỗi số parse; còn lại base64
    integer big-endian có dấu / 10^scale; lỗi hoặc rỗng -> None. Không bao giờ trả 0 thay cho None."""
    if value is None or isinstance(value, bool):
        return None
    if isinstance(value, (int, float)):
        return float(value)
    text = str(value)
    if text == "" or text in ("true", "false"):
        return None
    if PRICE_NUM_RE.match(text):
        return float(text)
    try:
        raw = base64.b64decode(text, validate=True)
    except (binascii.Error, ValueError):
        return None
    if not raw:
        return None
    return int.from_bytes(raw, "big", signed=True) / (10 ** scale)
# END_PRICE_PURE

DECODE = {scale: F.udf(lambda value, scale=scale: decode_price(value, scale), T.DoubleType()) for scale in (2, 4)}

def _require_types(df, expected, name):
    """Chặn khi build thiếu cột hoặc kiểu lệch Field Mapping L1. Không cast.

    Input : df — DataFrame L1; expected — {cột: simpleString}; name — tên bảng
    Output: None
    Ví dụ : _require_types(products, PRODUCT_TYPES, "slv_pn_products")
    """
    got = {field.name: field.dataType.simpleString() for field in df.schema.fields}
    missing = [col for col in expected if col not in got]
    bad = [f"{col}: {got[col]}" for col in expected if col in got and got[col] != expected[col]]
    if missing or bad:
        raise ConfigError(f"{name}: thiếu {missing}; lệch kiểu {bad}")


def _spec():
    return {
        "trg_tbl": "slv_pn_product_sale",
        "inputs": {
            "products": "slv_pn_products",
            "posts": "slv_pn_product_posts",
            "variants": "slv_pn_product_variants",
        },
        "required_inputs": ("products", "posts", "variants"),
        "block_checks": (("is_bookable_null", "is_bookable IS NULL"),),
        "pk": ("product_id",),
        "deleted_flag": "_is_deleted",
        "write_mode": "MERGE",
    }


def build(ctx):
    products, posts, variants = ctx.input("products"), ctx.input("posts"), ctx.input("variants")
    _require_types(products, PRODUCT_TYPES, "slv_pn_products")
    _require_types(posts, POST_TYPES, "slv_pn_product_posts")
    _require_types(variants, VARIANT_TYPES, "slv_pn_product_variants")
    live = posts.where(F.col("deleted") == F.lit(False))
    groups = live.groupBy("external_id").agg(F.count(F.lit(1)).alias("_n"))
    unique = (live.join(groups.where(F.col("_n") == 1), "external_id")
              .select(F.col("external_id").alias("_ext"), F.col("id").alias("product_post_id"),
                      "original_price", "marketing_price", "markup_percent"))
    ok = (F.col("status") == F.lit("active")) & (F.col("deleted") == F.lit(False))
    book = variants.groupBy(F.col("product_id").alias("_pid")).agg(F.max(F.when(ok, 1).otherwise(0)).alias("_ok"))
    base = products.select(
        F.col("id").alias("product_id"),
        F.get_json_object(F.col("metadata"), "$.holiday_product_post_id").alias("_ext"),
        F.col("deleted").alias(FLAG))
    rows = (base.join(unique, "_ext", "left").join(book, F.col("product_id") == F.col("_pid"), "left")
            .select(
                "product_id", "product_post_id",
                DECODE[4](F.col("original_price")).alias("original_price"),
                DECODE[4](F.col("marketing_price")).alias("marketing_price"),
                DECODE[2](F.col("markup_percent")).alias("markup_percent"),
                F.coalesce(F.col("_ok") == 1, F.lit(False)).alias("is_bookable"), FLAG,
                "_ext", F.col("original_price").alias("_rp"), F.col("marketing_price").alias("_rm"),
                F.col("markup_percent").alias("_rk")))
    rows = rows.persist(StorageLevel.MEMORY_AND_DISK)
    try:
        stat = rows.agg(
            F.sum(F.when(F.col("_ext").isNotNull() & F.col("product_post_id").isNull(), 1).otherwise(0)).alias("none"),
            F.sum(
                F.when(F.col("_rp").isNotNull() & F.col("original_price").isNull(), 1).otherwise(0)
                + F.when(F.col("_rm").isNotNull() & F.col("marketing_price").isNull(), 1).otherwise(0)
                + F.when(F.col("_rk").isNotNull() & F.col("markup_percent").isNull(), 1).otherwise(0)
            ).alias("bad"),
        ).first()
        ctx.warn("post_match_none", stat["none"])
        ctx.warn("price_decode_null", stat["bad"])
        referenced = rows.where(F.col("_ext").isNotNull()).select("_ext").distinct()
        many = groups.where(F.col("_n") > 1).select(F.col("external_id").alias("_ext"))
        ctx.warn("post_match_many", referenced.join(many, "_ext", "left_semi").count())
        return rows.select(*OUT_COLS, FLAG)
    finally:
        rows.unpersist()


SPEC = NodeSpec(
    trg_tbl="slv_pn_product_sale",
    inputs={"products": "slv_pn_products", "posts": "slv_pn_product_posts", "variants": "slv_pn_product_variants"},
    pk=("product_id",),
    build=build,
    required_inputs=("products", "posts", "variants"),
    block_checks=(("is_bookable_null", "is_bookable IS NULL"),),
    deleted_flag=FLAG,
    write_mode="MERGE",
)


In [ ]:
RES = run_node(SPEC, p_exec_id, p_run_id, p_pinned_json, p_mode, p_force, p_pl_name, p_trg_fq, p_history_limit)


In [ ]:
notebookutils.notebook.exit(finish_node(RES))
